In [0]:
# Notebook: 04_Model_Registration
import mlflow
from mlflow.tracking import MlflowClient

# Get the run ID from the previous training notebook run
# Option 1: Pass it as a parameter if running via Jobs/Workflows
# Option 2: Find the last successful run programmatically (can be fragile)
# For example purposes, let's assume it's passed as a widget or retrieved
# dbutils.widgets.text("train_run_id", "")
# train_run_id = dbutils.widgets.get("train_run_id")

# Or find the last run of the training notebook (adjust notebook path)
# runs = mlflow.search_runs(experiment_ids=mlflow.get_experiment_by_name("/Users/your_user@domain.com/03_Model_Training").experiment_id, order_by=["start_time DESC"], max_results=1)
# train_run_id = runs.iloc[0]['run_id'] if not runs.empty else None

In [0]:
# !! MANUALLY SET run_id if running interactively after step 4 !!
train_run_id = "3d42b40eda6e41a6b0ad4c669570f018" # Replace this!

if not train_run_id:
     dbutils.notebook.exit("Training Run ID not provided or found.")

# --- Model Registration ---
client = MlflowClient()
model_name = "AdventureWorksSalesPredictor" # Choose a descriptive name
model_artifact_path = "model" # Must match artifact_path in fs.log_model / mlflow.sklearn.log_model
model_uri = f"runs:/{train_run_id}/{model_artifact_path}"

print(f"Registering model '{model_name}' from URI: {model_uri}")

In [0]:
try:
    # Register the model
    registered_model_info = mlflow.register_model(
        model_uri=model_uri,
        name=model_name
    )
    print(f"Model registered: Name='{registered_model_info.name}', Version='{registered_model_info.version}'")

    # (Optional) Add description to the registered model version
    client.update_model_version(
        name=model_name,
        version=registered_model_info.version,
        description=f"Random Forest model trained on AdventureWorks data. Run ID: {train_run_id}."
    )

    # print(f"Transitioned model version {registered_model_info.version} to Staging.")

    dbutils.notebook.exit(f"{model_name}/{registered_model_info.version}")

except Exception as e:
    print(f"Error registering model: {e}")
    # dbutils.notebook.exit("Model registration failed.")

In [0]:
# Create an alias for the registered model
model_alias = "staging"

try:
    client.set_registered_model_alias(
        name=model_name,
        alias=model_alias,
        version=registered_model_info.version
    )
    print(f"Alias '{model_alias}' created for model '{model_name}' version '{registered_model_info.version}'")
except Exception as e:
    print(f"Error creating model alias: {e}")